In [1]:
import os
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Load data

In [2]:
def read_parquet(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    user_chunk_files = [file for file in files if 'user_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    user_chunk_df = pl.concat([pl.read_parquet(file) for file in user_chunk_files]) if user_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return user_chunk_df

In [3]:
def read_parquet_item(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    item_chunk_files = [file for file in files if 'item_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    item_chunk_df = pl.concat([pl.read_parquet(file) for file in item_chunk_files]) if item_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return item_chunk_df

In [4]:
def read_parquet_purchase(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    purchase_chunk_files = [file for file in files if 'purchase_history_daily_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    purchase_chunk_df = pl.concat([pl.read_parquet(file) for file in purchase_chunk_files]) if purchase_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return purchase_chunk_df

In [5]:
trans_df = read_parquet_purchase(".././preprocessed-dataset")
trans_df

item_id,quantity,customer_id,created_date,location,price,log_price,discount_rate,channel,payment_bucket,time_between_purchases,month,seasonal_trend,product_engagement_level
str,i32,i32,datetime[μs],i32,f64,f64,f64,str,str,duration[μs],i8,str,str
"""7115000000004""",1,5254214,2024-12-24 18:17:01.027,656,49000.0,10.799596,0.0,"""In-Store""","""qr""",316d 20h 50m 27s 623ms,12,"""Winter""","""High"""
"""0029130000030""",1,7573232,2024-12-24 19:28:01.870,143,69000.0,11.141876,0.0,"""In-Store""","""cash""",203d 1h 54m 54s 827ms,12,"""Winter""","""High"""
"""3496000000053""",2,8187418,2024-12-24 19:50:43.760,213,75000.0,11.225257,0.0,"""In-Store""","""wallet""",1m 4s 537ms,12,"""Winter""","""High"""
"""2700000000002""",2,8187418,2024-12-24 19:49:39.223,213,58500.0,10.976799,0.1,"""In-Store""","""wallet""",1m 4s 537ms,12,"""Winter""","""High"""
"""0029110000036""",1,6931560,2024-12-28 09:49:33.780,590,89000.0,11.396403,0.10101,"""Android""","""wallet""",354d 16h 9m 39s 93ms,12,"""Winter""","""High"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""0175000000011""",1,2083231,2024-12-09 17:37:34.450,697,69000.0,11.141876,0.0,"""In-Store""","""wallet""",325d 17h 52m 41s 606ms,12,"""Winter""","""High"""
"""5902000000002""",1,8141777,2024-12-09 08:01:39.743,353,115000.0,11.652696,0.0,"""In-Store""","""cash""",0µs,12,"""Winter""","""High"""
"""0175000000007""",1,8066612,2024-12-09 19:22:15.857,940,79000.0,11.277216,0.0,"""In-Store""","""cash""",17d 8h 6m 50s 280ms,12,"""Winter""","""High"""


In [6]:
item_df = read_parquet_purchase(".././preprocessed-dataset")
item_df.head()

item_id,quantity,customer_id,created_date,location,price,log_price,discount_rate,channel,payment_bucket,time_between_purchases,month,seasonal_trend,product_engagement_level
str,i32,i32,datetime[μs],i32,f64,f64,f64,str,str,duration[μs],i8,str,str
"""7115000000004""",1,5254214,2024-12-24 18:17:01.027,656,49000.0,10.799596,0.0,"""In-Store""","""qr""",316d 20h 50m 27s 623ms,12,"""Winter""","""High"""
"""0029130000030""",1,7573232,2024-12-24 19:28:01.870,143,69000.0,11.141876,0.0,"""In-Store""","""cash""",203d 1h 54m 54s 827ms,12,"""Winter""","""High"""
"""3496000000053""",2,8187418,2024-12-24 19:50:43.760,213,75000.0,11.225257,0.0,"""In-Store""","""wallet""",1m 4s 537ms,12,"""Winter""","""High"""
"""2700000000002""",2,8187418,2024-12-24 19:49:39.223,213,58500.0,10.976799,0.1,"""In-Store""","""wallet""",1m 4s 537ms,12,"""Winter""","""High"""
"""0029110000036""",1,6931560,2024-12-28 09:49:33.780,590,89000.0,11.396403,0.10101,"""Android""","""wallet""",354d 16h 9m 39s 93ms,12,"""Winter""","""High"""


In [7]:
# Cell 1
from sklearn.cluster import KMeans, MiniBatchKMeans

# Giữ nguyên bản gốc để join trả về
tx_orig = trans_df.clone()
item = item_df.clone()

In [9]:
# Cell 2 (sửa lỗi dtype)
tx = tx_orig.clone()

formats = [
    "%Y-%m-%d %H:%M:%S",
    "%Y-%m-%dT%H:%M:%S",
    "%Y-%m-%d",
    "%d/%m/%Y %H:%M:%S",
    "%d/%m/%Y"
]

parsed_ok = False
for fmt in formats:
    try:
        parsed = tx.with_columns([
            pl.col("created_date").str.strptime(pl.Datetime, fmt, strict=False).alias("created_date_parsed")
        ])
    except Exception:
        continue

    # dùng to_numpy()[0,0] để lấy scalar (tương thích nhiều version)
    nulls = parsed.select(pl.col("created_date_parsed").is_null().sum()).to_numpy()[0,0]
    if nulls < parsed.height * 0.5:
        tx = parsed.with_columns([pl.col("created_date_parsed").alias("created_date")]).drop("created_date_parsed")
        parsed_ok = True
        break

if not parsed_ok:
    tx = tx.with_columns([pl.col("created_date").cast(pl.Datetime).alias("created_date")])

# In kiểu dữ liệu và vài mẫu để kiểm tra (cách tương thích với mọi phiên bản)
dtype = tx.schema.get("created_date")
print("created_date dtype (schema):", dtype)
print("Một vài giá trị created_date (head):")
print(tx.select("created_date").head(5))


created_date dtype (schema): Datetime(time_unit='us', time_zone=None)
Một vài giá trị created_date (head):
shape: (5, 1)
┌─────────────────────────┐
│ created_date            │
│ ---                     │
│ datetime[μs]            │
╞═════════════════════════╡
│ 2024-12-24 18:17:01.027 │
│ 2024-12-24 19:28:01.870 │
│ 2024-12-24 19:50:43.760 │
│ 2024-12-24 19:49:39.223 │
│ 2024-12-28 09:49:33.780 │
└─────────────────────────┘


In [10]:
# Cell 3
# đảm bảo các cột numeric trước khi tính
tx = tx.with_columns([
    pl.col("price").cast(pl.Float64).alias("price_f"),
    pl.col("quantity").cast(pl.Float64).alias("quantity_f"),
    pl.col("discount_rate").cast(pl.Float64).alias("discount_rate_f")
])

# Nếu discount_rate null thì coi = 0
tx = tx.with_columns([
    (pl.col("price_f") * pl.col("quantity_f") * (1 - pl.col("discount_rate_f").fill_null(0.0))).alias("line_amount")
])

# Kiểm tra tóm tắt
print("Tổng dòng, sample line_amount:")
print(tx.select(["price_f","quantity_f","discount_rate_f","line_amount"]).head(5))


Tổng dòng, sample line_amount:
shape: (5, 4)
┌─────────┬────────────┬─────────────────┬─────────────┐
│ price_f ┆ quantity_f ┆ discount_rate_f ┆ line_amount │
│ ---     ┆ ---        ┆ ---             ┆ ---         │
│ f64     ┆ f64        ┆ f64             ┆ f64         │
╞═════════╪════════════╪═════════════════╪═════════════╡
│ 49000.0 ┆ 1.0        ┆ 0.0             ┆ 49000.0     │
│ 69000.0 ┆ 1.0        ┆ 0.0             ┆ 69000.0     │
│ 75000.0 ┆ 2.0        ┆ 0.0             ┆ 150000.0    │
│ 58500.0 ┆ 2.0        ┆ 0.1             ┆ 105300.0    │
│ 89000.0 ┆ 1.0        ┆ 0.10101         ┆ 80010.10101 │
└─────────┴────────────┴─────────────────┴─────────────┘


In [11]:
# Cell 4
# Bỏ dòng line_amount null trước khi group (nếu có)
tx_filtered = tx.filter(pl.col("line_amount").is_not_null())

purchase_amount = (
    tx_filtered
      .group_by(["customer_id", "created_date"])
      .agg(
          pl.col("line_amount").sum().alias("transaction_amount")  # tổng tiền cho 1 lần mua
      )
)

print("Số lần mua (rows):", purchase_amount.height)
print(purchase_amount.head(5))


Số lần mua (rows): 16196081
shape: (5, 3)
┌─────────────┬─────────────────────────┬────────────────────┐
│ customer_id ┆ created_date            ┆ transaction_amount │
│ ---         ┆ ---                     ┆ ---                │
│ i32         ┆ datetime[μs]            ┆ f64                │
╞═════════════╪═════════════════════════╪════════════════════╡
│ 6897402     ┆ 2024-03-10 10:06:06.097 ┆ 457214.444444      │
│ 6672843     ┆ 2024-01-25 13:01:47.377 ┆ 338000.0           │
│ 5908766     ┆ 2024-09-20 21:14:23.217 ┆ 938000.0           │
│ 4810634     ┆ 2024-10-25 19:50:24.693 ┆ 972872.093023      │
│ 5011133     ┆ 2024-10-11 18:02:07.890 ┆ 312000.0           │
└─────────────┴─────────────────────────┴────────────────────┘


In [12]:
# Cell 5
cust_avg_amount = (
    purchase_amount
      .group_by("customer_id")
      .agg(pl.col("transaction_amount").mean().alias("avg_transaction_amount_per_purchase"))
)

print("Số khách có avg:", cust_avg_amount.height)
print(cust_avg_amount.head(5))


Số khách có avg: 2442306
shape: (5, 2)
┌─────────────┬─────────────────────────────────┐
│ customer_id ┆ avg_transaction_amount_per_pur… │
│ ---         ┆ ---                             │
│ i32         ┆ f64                             │
╞═════════════╪═════════════════════════════════╡
│ 4079118     ┆ 511298.91851                    │
│ 7453991     ┆ 79000.0                         │
│ 5417670     ┆ 155317.897315                   │
│ 7104422     ┆ 396818.84058                    │
│ 7493327     ┆ 73500.0                         │
└─────────────┴─────────────────────────────────┘


In [13]:
# Cell 6
# Nếu cust quá lớn, dùng sample để fit, sau đó predict trên full set
n_customers = int(cust_avg_amount.height)
print("Số khách:", n_customers)

# Chuẩn bị dữ liệu numpy (1D -> 2D)
X_full = cust_avg_amount.select("avg_transaction_amount_per_purchase").to_numpy().astype(float).reshape(-1,1)

# Chọn chiến lược: nếu quá nhiều khách, fit trên sample bằng MiniBatchKMeans
SAMPLE_FOR_FIT = 50000  # fit trên 50k nếu lớn hơn
if n_customers == 0:
    print("Không có khách để phân cụm.")
else:
    if n_customers > SAMPLE_FOR_FIT:
        rng = np.random.default_rng(42)
        idx = rng.choice(n_customers, size=SAMPLE_FOR_FIT, replace=False)
        X_sample = X_full[idx]
        km = MiniBatchKMeans(n_clusters=3, random_state=42, batch_size=2048, n_init=10)
        km.fit(X_sample)
    else:
        km = KMeans(n_clusters=3, random_state=42, n_init=10)
        km.fit(X_full)

    # predict label cho toàn bộ
    labels_full = km.predict(X_full)
    centers = km.cluster_centers_.ravel()
    print("Centers (raw):", centers)


Số khách: 2442306
Centers (raw): [ 606405.4248947   208373.47845726 1493919.46636376]


In [14]:
# Cell 7
order = np.argsort(centers)  # chỉ số cluster theo tâm tăng dần
rank_map = { raw: rank for rank, raw in enumerate(order) }  # raw_id -> rank 0..2
cluster_rank = np.vectorize(rank_map.get)(labels_full)
label_map = {0: "Bình dân", 1: "Trung cấp", 2: "Cao cấp"}
segment_name = np.vectorize(label_map.get)(cluster_rank)

# Thêm cột vào cust_avg_amount
cust_avg_amount = cust_avg_amount.with_columns([
    pl.Series("cluster_id_raw", labels_full.astype(int)),
    pl.Series("cluster_rank", cluster_rank.astype(int)),
    pl.Series("segment_name", segment_name)
])

# Xem tóm tắt
print(cust_avg_amount.head(5))


shape: (5, 5)
┌─────────────┬─────────────────────────────────┬────────────────┬──────────────┬──────────────┐
│ customer_id ┆ avg_transaction_amount_per_pur… ┆ cluster_id_raw ┆ cluster_rank ┆ segment_name │
│ ---         ┆ ---                             ┆ ---            ┆ ---          ┆ ---          │
│ i32         ┆ f64                             ┆ i32            ┆ i32          ┆ str          │
╞═════════════╪═════════════════════════════════╪════════════════╪══════════════╪══════════════╡
│ 4079118     ┆ 511298.91851                    ┆ 0              ┆ 1            ┆ Trung cấp    │
│ 7453991     ┆ 79000.0                         ┆ 1              ┆ 0            ┆ Bình dân     │
│ 5417670     ┆ 155317.897315                   ┆ 1              ┆ 0            ┆ Bình dân     │
│ 7104422     ┆ 396818.84058                    ┆ 1              ┆ 0            ┆ Bình dân     │
│ 7493327     ┆ 73500.0                         ┆ 1              ┆ 0            ┆ Bình dân     │
└─────────────┴─

In [15]:
# Cell 8
trans_df_augmented = tx_orig.join(
    cust_avg_amount.select(["customer_id", "avg_transaction_amount_per_purchase", "segment_name"]),
    on="customer_id",
    how="left"
)

print("Số dòng trans_df_augmented:", trans_df_augmented.height)
print(trans_df_augmented.select(["customer_id","item_id","created_date","avg_transaction_amount_per_purchase","segment_name"]).head(10))


Số dòng trans_df_augmented: 35729825
shape: (10, 5)
┌─────────────┬───────────────┬─────────────────────────┬───────────────────────────┬──────────────┐
│ customer_id ┆ item_id       ┆ created_date            ┆ avg_transaction_amount_pe ┆ segment_name │
│ ---         ┆ ---           ┆ ---                     ┆ r_pur…                    ┆ ---          │
│ i32         ┆ str           ┆ datetime[μs]            ┆ ---                       ┆ str          │
│             ┆               ┆                         ┆ f64                       ┆              │
╞═════════════╪═══════════════╪═════════════════════════╪═══════════════════════════╪══════════════╡
│ 5254214     ┆ 7115000000004 ┆ 2024-12-24 18:17:01.027 ┆ 466866.614601             ┆ Trung cấp    │
│ 7573232     ┆ 0029130000030 ┆ 2024-12-24 19:28:01.870 ┆ 679508.333333             ┆ Trung cấp    │
│ 8187418     ┆ 3496000000053 ┆ 2024-12-24 19:50:43.760 ┆ 596511.904847             ┆ Trung cấp    │
│ 8187418     ┆ 2700000000002 ┆ 2024-12

In [16]:
# Cell 9
cluster_summary = (
    cust_avg_amount
      .group_by(["cluster_rank","segment_name"])
      .agg([
          pl.len().alias("n_customers"),
          pl.col("avg_transaction_amount_per_purchase").mean().alias("center_empirical"),
          pl.col("avg_transaction_amount_per_purchase").min().alias("min"),
          pl.col("avg_transaction_amount_per_purchase").quantile(0.5).alias("median"),
          pl.col("avg_transaction_amount_per_purchase").max().alias("max"),
      ])
      .sort("cluster_rank")
)

print(cluster_summary)


shape: (3, 7)
┌──────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ cluster_rank ┆ segment_nam ┆ n_customers ┆ center_empi ┆ min         ┆ median      ┆ max         │
│ ---          ┆ e           ┆ ---         ┆ rical       ┆ ---         ┆ ---         ┆ ---         │
│ i32          ┆ ---         ┆ u32         ┆ ---         ┆ f64         ┆ f64         ┆ f64         │
│              ┆ str         ┆             ┆ f64         ┆             ┆             ┆             │
╞══════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ 0            ┆ Bình dân    ┆ 1615827     ┆ 207202.8763 ┆ 0.5982      ┆ 204571.4285 ┆ 407387.8521 │
│              ┆             ┆             ┆ 38          ┆             ┆ 71          ┆ 13          │
│ 1            ┆ Trung cấp   ┆ 743065      ┆ 604492.5798 ┆ 407390.7644 ┆ 563567.6046 ┆ 1.050155e6  │
│              ┆             ┆             ┆ 59          ┆ 4           ┆ 95  